# Laser Off Code

In [ ]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

# Station

In [12]:
snap = station.snapshot(update=True) # <- updates parameters in station

In [13]:
verticalsnap = snap['instruments']['mso5']['submodules']['channels']['channels']['mso5_ch1']['parameters'].keys()

In [14]:
horizontalsnap = snap['instruments']['mso5']['parameters'].keys()

# Import

In [1]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
initialise_or_create_database_at("./2026-03-10_SNSPD4.db")
from functions import osc_set_standard, osc_check_standard, capture_trace, capture_trace_simple
from functions import snspd_dark_counts
import snspd4

# Set up experiment
exp_name = 'SNSPD4_23_03_2026'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

Logging hadn't been started.
Activating auto-logging. Current session state plus future input saved.
Filename       : C:\Users\QNL\.qcodes\logs\command_history.log
Mode           : append
Output logging : True
Raw input log  : False
Timestamping   : True
State          : active
Qcodes Logfile : C:\Users\QNL\.qcodes\logs\260410-30368-qcodes.log
Experiment loaded. Last ID no: 453


In [2]:
station = Station(config_file="friesland.yaml")

In [3]:
dmm = station.load_instrument("dmm", revive_instance=True)

Connected to: Agilent Technologies 34410A (serial:MY47027892, firmware:2.35-2.35-0.09-46-09) in 0.11s


In [4]:
yoko = station.load_instrument("yoko", revive_instance=True)

Connected to: YOKOGAWA GS210 (serial:91T928105, firmware:2.02) in 0.03s


In [5]:
laser = station.load_instrument("laser", revive_instance=True)

2026-04-10 15:57:48,308 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ D:\SNSPD\SNSPD2\PPCL550.py:4: QCoDeSDeprecationWarning: The `qcodes.utils.helpers` module is deprecated. Please consult the api documentation at https://microsoft.github.io/Qcodes/api/index.html for alternatives.
  from qcodes.utils.helpers import create_on_off_val_mapping

2026-04-10 15:57:48,325 ¦ qcodes.instrument.instrument_base.com.visa ¦ ERROR ¦ visa ¦ _connect_and_handle_error ¦ 222 ¦ [laser(PPCL550)] Could not connect at ASRL13
Traceback (most recent call last):
  File "C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\qcodes\instrument\visa.py", line 218, in _connect_and_handle_error
    visa_handle, visabackend, resource_manager = self._open_resource(
                                                 ~~~~~~~~~~~~~~~~~~~^
        address, visalib
        ^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\qcodes\instrument\visa.py", line 246, in _open_resou

VisaIOError: VI_ERROR_RSRC_BUSY (-1073807246): The resource is valid, but VISA cannot currently access it.

In [7]:
MS = station.load_instrument("mso5", revive_instance=True)

2026-04-10 15:58:06,820 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ D:\SNSPD\SNSPD2\MSO5.py:5: QCoDeSDeprecationWarning: The `qcodes.instrument.base` module is deprecated. Please consult the api documentation at https://microsoft.github.io/Qcodes/api/index.html for alternatives.
  from qcodes.instrument.base import InstrumentBase



In [12]:
pm100usb = station.load_instrument("pm100usb", revive_instance=True)

In [13]:
pms120 = station.load_instrument("pms120", revive_instance=True)

In [14]:
tc = station.load_instrument("fridge", revive_instance=True)

In [15]:
p_att = station.load_instrument("dmm_keithley", revive_instance=True) # excluding from snapshot because none of the parameters work anyway

# Parameters

In [16]:
device_name_1 = 'Line 1 R7C6'
device_name_2 = 'Line 2 Old Device'
att_screw = 'VOA50PM' 
att_blue = 'V1550PA'
# Beam splitter ratio measured ...? 
bs90 = 0.9118079062146573 
bs10 = 0.08819209378534265
h_time = 100e-3 # full length trace for dark counts
h_pos = 0
v_scale=150e-3 # must check for clipping 

# Dark Counts

In [21]:
osc_set_standard(MS, v_scale=v_scale, h_time=h_time, h_pos=h_pos)

In [22]:
snspd_dark_counts(MS, dmm, yoko, device_name=device_name_1, n_captures=10, interval=1, currents=[12e-6], station=station)

Starting experimental run with id: 450. 
450
This acquisition will take 10s
15 24


In [21]:
ID = 450 
data = load_by_id(450).get_parameter_data()
CR1 = data['CR1']['CR1']
CR2 = data['CR2']['CR2']
counts1 = data['counts1']['counts1']
counts2 = data['counts2']['counts2']

In [22]:
counts1

array([0., 1., 0., 1., 1., 3., 1., 2., 1., 1.])

In [17]:
osc_set_standard(MS, v_scale=v_scale, h_time=h_time, h_pos=h_pos)
snspd_dark_counts(MS, dmm, yoko, device_name=device_name_1, n_captures=3000, interval=1, currents=[12e-6], station=station)

2026-04-10 15:58:29,931 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 454. 
454
This acquisition will take 3000s
15 58


In [39]:
osc_set_standard(MS, v_scale=v_scale, h_time=h_time, h_pos=h_pos)
snspd_dark_counts(MS, dmm, yoko, device_name=device_name_1, n_captures=3000, interval=1, currents=[12e-6], station=station)

2026-04-10 18:58:58,586 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Starting experimental run with id: 455. 
455
This acquisition will take 3000s
18 59
